In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("skill-alias-builder")
    .master("spark://spark-master:7077")
    .config("spark.sql.adaptive.enabled", "false")          # fix version mismatch
    .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1")
    .config("spark.sql.catalog.nessie.ref", "test")
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin")
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password")
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .getOrCreate()
)

spark

In [2]:
df = spark.table("nessie.silver.jobs")

tech = (
    df.select(F.explode("tech_skills").alias("raw_skill"))
    .filter(F.col("raw_skill").isNotNull() & (F.col("raw_skill") != ""))
    .withColumn("skill_type", F.lit("tech"))
)

soft = (
    df.select(F.explode("soft_skills").alias("raw_skill"))
    .filter(F.col("raw_skill").isNotNull() & (F.col("raw_skill") != ""))
    .withColumn("skill_type", F.lit("soft"))
)

# Normalize trước khi group: lowercase + strip để merge case variants
all_skills = (
    tech.union(soft)
    .withColumn("raw_skill_norm", F.trim(F.lower(F.col("raw_skill"))))
    # Giữ raw_skill gốc phổ biến nhất để dễ đọc khi hand-craft
    .groupBy("raw_skill_norm", "skill_type")
    .agg(
        F.count("*").alias("count"),
        F.first("raw_skill").alias("raw_skill_display")  # form gốc để tham khảo
    )
    .orderBy(F.col("count").desc())
)

print(f"Unique skills (sau normalize): {all_skills.count()}")
all_skills.show(20, truncate=False)

Unique skills (sau normalize): 40780
+-----------------+----------+-----+-----------------+
|raw_skill_norm   |skill_type|count|raw_skill_display|
+-----------------+----------+-----+-----------------+
|giao tiếp        |soft      |32167|giao tiếp        |
|excel            |tech      |12956|Excel            |
|làm việc nhóm    |soft      |11239|làm việc nhóm    |
|tiếng anh        |soft      |10064|Tiếng Anh        |
|làm việc độc lập |soft      |9291 |làm việc độc lập |
|đàm phán         |soft      |7509 |đàm phán         |
|word             |tech      |6510 |Word             |
|giải quyết vấn đề|soft      |5990 |giải quyết vấn đề|
|thuyết phục      |soft      |5789 |thuyết phục      |
|xử lý tình huống |soft      |5206 |xử lý tình huống |
|phân tích        |soft      |5051 |phân tích        |
|english          |tech      |4364 |English          |
|bán hàng         |soft      |4361 |bán hàng         |
|communication    |soft      |4045 |communication    |
|lập kế hoạch     |soft     

In [3]:
import pandas as pd

pdf = all_skills.toPandas()
pdf["cumsum"] = pdf["count"].cumsum()
pdf["cum_pct"] = pdf["cumsum"] / pdf["count"].sum() * 100

for threshold in [50, 70, 80, 90, 95]:
    n = (pdf["cum_pct"] <= threshold).sum()
    print(f"Top {n:4d} skills cover {threshold}% tổng occurrences")

Top  372 skills cover 50% tổng occurrences
Top 2634 skills cover 70% tổng occurrences
Top 6021 skills cover 80% tổng occurrences
Top 12880 skills cover 90% tổng occurrences
Top 20368 skills cover 95% tổng occurrences


In [4]:
import pandas as pd

df_silver = spark.table("nessie.silver.jobs")

tech = (
    df_silver.select(F.explode("tech_skills").alias("raw_skill"))
    .filter(F.col("raw_skill").isNotNull() & (F.col("raw_skill") != ""))
    .withColumn("skill_type", F.lit("tech"))
)
soft = (
    df_silver.select(F.explode("soft_skills").alias("raw_skill"))
    .filter(F.col("raw_skill").isNotNull() & (F.col("raw_skill") != ""))
    .withColumn("skill_type", F.lit("soft"))
)

all_skills = (
    tech.union(soft)
    .withColumn("raw_skill_norm", F.trim(F.lower(F.col("raw_skill"))))
    .groupBy("raw_skill_norm", "skill_type")
    .agg(
        F.count("*").alias("freq"),
        F.first("raw_skill").alias("raw_skill_display")
    )
    .orderBy(F.col("freq").desc())
)

skills_pd = all_skills.toPandas()
print(f"Total (raw, type) pairs: {len(skills_pd)}")
print(skills_pd["skill_type"].value_counts())

Total (raw, type) pairs: 40780
skill_type
tech    34741
soft     6039
Name: count, dtype: int64


In [5]:
import pandas as pd

CSV_PATH = "/home/jovyan/notebooks/skill_freq_6k.csv"

pdf_filled = pd.read_csv(CSV_PATH, on_bad_lines='skip')
print(f"Loaded: {len(pdf_filled)} rows từ CSV")
print(f"Columns: {list(pdf_filled.columns)}")
print(f"Empty canonical: {pdf_filled['canonical'].isna().sum()}")
print(f"\nSkill type distribution:")
print(pdf_filled["skill_type"].value_counts())

Loaded: 5994 rows từ CSV
Columns: ['skill_type', 'raw_skill_norm', 'raw_skill_display', 'count', 'canonical', 'note']
Empty canonical: 0

Skill type distribution:
skill_type
soft    2999
tech    2995
Name: count, dtype: int64


In [6]:
import json

def smart_title(s):
    if not isinstance(s, str): return s
    words = s.split()
    return " ".join(
        w if (w.isupper() or any(c.isdigit() for c in w))
        else w.capitalize()
        for w in words
    )

alias_dict = {}
skipped = 0

for _, row in pdf_filled.iterrows():
    canonical = str(row["canonical"]).strip() if pd.notna(row["canonical"]) else ""
    if not canonical:
        skipped += 1
        continue

    composite = f"{row['raw_skill_norm'].strip().lower()}|{row['skill_type'].strip()}"
    alias_dict[composite] = {
        "canonical": smart_title(canonical),
        "type": row["skill_type"].strip()
    }

output_path = "/home/jovyan/notebooks/skill_alias_bootstrap.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(alias_dict, f, ensure_ascii=False, indent=2)

print(f"Saved {len(alias_dict)} entries → {output_path}")
print(f"Skipped (empty canonical): {skipped}")
print(f"Tech: {sum(1 for v in alias_dict.values() if v['type']=='tech')}")
print(f"Soft: {sum(1 for v in alias_dict.values() if v['type']=='soft')}")
print("\nSample:")
for k, v in list(alias_dict.items())[:5]:
    print(f"  {k!r:40s} → {v}")

Saved 5988 entries → /home/jovyan/notebooks/skill_alias_bootstrap.json
Skipped (empty canonical): 0
Tech: 2995
Soft: 2993

Sample:
  'giao tiếp|soft'                         → {'canonical': 'Giao Tiếp', 'type': 'soft'}
  'làm việc nhóm|soft'                     → {'canonical': 'Làm Việc Nhóm', 'type': 'soft'}
  'tiếng anh|soft'                         → {'canonical': 'Tiếng Anh', 'type': 'soft'}
  'làm việc độc lập|soft'                  → {'canonical': 'Làm Việc Độc Lập', 'type': 'soft'}
  'đàm phán|soft'                          → {'canonical': 'Đàm Phán', 'type': 'soft'}
